In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
%%capture
!pip install haversine
!pip install transformers
!pip install sentencepiece
!pip install accelerate

In [ ]:
import polars as pl
import geopy.geocoders as geocoders
from geopy.exc import GeocoderTimedOut
from geopy.extra.rate_limiter import RateLimiter
import numpy as np
from geopy.distance import geodesic
import haversine as hs
import datetime as dt
from tqdm import tqdm
import pandas as pd

import random
from tqdm import tqdm

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
########################################################################################################
device = torch.device( 'mps'  if torch.backends.mps.is_available() else \
                       'cuda' if torch.cuda.is_available() else 'cpu' )
########################################################################################################
STATES = {
    'AK': 'Alaska', 'AL': 'Alabama', 'AR': 'Arkansas', 'AZ': 'Arizona', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DC': 'District of Columbia', 'DE': 'Delaware', 'FL': 'Florida',
    'GA': 'Georgia', 'HI': 'Hawaii', 'IA': 'Iowa', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'MA': 'Massachusetts', 'MD': 'Maryland',
    'ME': 'Maine', 'MI': 'Michigan', 'MN': 'Minnesota', 'MO': 'Missouri', 'MS': 'Mississippi',
    'MT': 'Montana', 'NC': 'North Carolina', 'ND': 'North Dakota', 'NE': 'Nebraska', 'NH': 'New Hampshire',
    'NJ': 'New Jersey', 'NM': 'New Mexico', 'NV': 'Nevada', 'NY': 'New York', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VA': 'Virginia',
    'VT': 'Vermont', 'WA': 'Washington', 'WI': 'Wisconsin', 'WV': 'West Virginia', 'WY': 'Wyoming' }
STATES = { k:v+' state, USA' for k,v in STATES.items() }
########################################################################################################
gl = dict(geolocator=geocoders.Nominatim(user_agent='to_coordinate', timeout=50))
def get_coordinate(address, \
                   geolocator=geocoders.Nominatim(user_agent='to_coordinate')):
    '''
    address (str):
        str containing "post_code city_name state_name country_code".
    '''
    if isinstance(address, str):
        try:
            gc = RateLimiter(geolocator.geocode, min_delay_seconds=0.02)
            loc = gc(address)
            if loc is not None:
                return (loc.latitude, loc.longitude)
        except GeocoderTimedOut:
            return get_coordinate(address, geolocator)
    return (999, 999)
def get_lat_long(series):
    lat, lng = {}, {}
    for m in tqdm(series, desc='getting coordinate...'):
        if m is not None:
            try:
                lat[m], lng[m] = get_coordinate(STATES[m] if m in STATES else m, **gl)
            except:
                lat[m], lng[m] = (999, 999)
    lat[None], lng[None] = (999, 999)
    return lat, lng
def proc_region_str(record):
    if record in STATES:
        return STATES[record]
    else:
        if record is None:
            return ''
        else:
            return record
########################################################################################################

card, user = pl.read_csv('./card.csv'), pl.read_csv('./user.csv')
test, train = pl.read_csv('./test.csv'), pl.read_csv('./train.csv')
print(f"Shared_columns(test, user) = {set(test.columns).intersection(set(user.columns))}, \nShared_columns(test, card) = {set(test.columns).intersection(set(card.columns))}")
df = train.join(card.join(user, on='user_id', how='left'), on=('user_id', 'card_id'), how='left')
df_test = test.join(card.join(user, on='user_id', how='left'), on=('user_id', 'card_id'), how='left')

mstate_lat, mstate_long = get_lat_long(df['merchant_state'].unique().append(df_test['merchant_state'].unique()))

prob_vars = [ 'state', 'merchant_state', 'zip', 'city', 'address', 'merchant_city', 'mcc' ]
# 'state', 'merchant_state', 'zip' has missing values in prob_vars. hence,
region_vars = [ 'city', 'merchant_city', 'mcc' ]
region_dummies = df[region_vars].vstack(df_test[region_vars]).to_dummies()
region_dummies, region_dummies_test = region_dummies[0:df.shape[0]], region_dummies[df.shape[0]:]



########################################################################################################
merchant_unique_id = set()
for ele in df.select( pl.when( pl.col('is_fraud?') == pl.lit(1) ).then( pl.col('merchant_id') ).otherwise(None).alias('address') ).unique()['address']:
    if ele is not None:
        merchant_unique_id.add(ele)
def pipeline(df):
    amt = lambda col: pl.col(col).str.replace("^\$", "").cast(pl.Float64)
    dummies = ['use_chip','card_brand','card_type','cards_issued']
    is_coordinate = lambda lat, lng: True if (lat is not None) and (lng is not None) and (abs(lat)<=90) and (abs(lng)<=180) else False
    df = df.with_columns( (pl.col('merchant_city')==pl.col('city')).cast(pl.Int64).alias('city_match'),
                           pl.col('merchant_state').map_dict(mstate_long).alias('mstate_long'),
                           pl.col('merchant_state').map_dict(mstate_lat).alias('mstate_lat'),
                           pl.col('acct_open_date').str.strptime(pl.Datetime, "%m/%Y").cast(pl.Float64)/1e+15,
                           pl.col('expires').str.strptime(pl.Datetime, "%m/%Y").cast(pl.Float64)/1e+15,
                           pl.col('gender').map_dict({'Male':1,'Female':0}),
                           pl.col('has_chip').map_dict({'YES':1,'NO':0}),
                          (pl.col('errors?')!='OK').cast(pl.Int64),
                           pl.when( pl.col('merchant_id').is_in(merchant_unique_id) ).then(pl.lit(1)).otherwise(pl.lit(0)),
                           pl.col('zip').fill_null(-1),
                           pl.col('card_id') + 1,
                           pl.col('year_pin_last_changed') - 1900,
                           pl.col('birth_year') - 1900,
                           amt('amount')/1e+3,
                           amt('total_debt')/1e+3,
                           amt('credit_limit')/1e+3,
                           amt('yearly_income_person')/1e+3,
                           amt('per_capita_income_zipcode')/1e+3 )\
           .with_columns( (pl.col('expires') - pl.col('acct_open_date')).alias('exp_min_aod') )\
           .hstack(df[dummies].to_dummies())\
           .drop(dummies)
    df = df.with_columns( pl.Series(name='merchant_user_dist',\
                                    values=( hs.haversine(ms_c, c) if is_coordinate(ms_c[0], ms_c[1]) else 0 \
                                             for ms_c, c in zip(df[['mstate_lat', 'mstate_long']].iter_rows(), df[['latitude', 'longitude']].iter_rows()) )) / 10000 )
    df = df.with_columns( pl.when( pl.col('mstate_lat')!=999 ).then(pl.col('mstate_lat')).otherwise(pl.lit(0)),
                          pl.when( pl.col('mstate_long')!=999 ).then(pl.col('mstate_long')).otherwise(pl.lit(0)),
                          pl.when( pl.col('merchant_user_dist')==0. ).then(pl.lit(1)).otherwise(pl.lit(0)).alias('merchant_dist_online') )
    return df
########################################################################################################


drop_vars = [ 'index', 'user_id' ]
pdf = pipeline(df).hstack(region_dummies).drop(prob_vars+drop_vars)
pdf_test = pipeline(df_test).hstack(region_dummies_test).drop(prob_vars+drop_vars)

dep_vars = [ 'is_fraud?' ]
train_df_frd = pdf.filter( pl.col('is_fraud?')==1 )
train_df_not_frd = pdf.filter( pl.col('is_fraud?')==0 )
dep_frd, indep_frd, dep_not_frd, indep_not_frd = train_df_frd[dep_vars], train_df_frd.drop(dep_vars), train_df_not_frd[dep_vars], train_df_not_frd.drop(dep_vars)
indep_vars = indep_frd.columns

In [20]:
# PyTorch ##############################################################################################
def random_row(dep_frd, indep_frd, dep_not_frd, indep_not_frd):
    frd_or_not = random.choices([0, 1], weights=(6, 4), k=1)[0]
    dep_df, indep_df = (dep_frd, indep_frd) if frd_or_not==1 else (dep_not_frd, indep_not_frd)
    row_num = random.randint(0, dep_df.shape[0])
    dep_row = dep_df[row_num]
    indep_row = indep_df[row_num]
    return dep_row, indep_row

def random_training_pair(dep_frd, indep_frd, dep_not_frd, indep_not_frd):
    dep_row, indep_row = random_row(dep_frd, indep_frd, dep_not_frd, indep_not_frd)
    # print(dep_row, indep_row) # .to_series().cast(pl.Float32), .transpose().to_series().cast(pl.Float32)
    dep_row_torch = torch.from_numpy(dep_row.to_numpy().squeeze(axis=1)).to(torch.long).to(device)
    indep_row_torch = torch.from_numpy(indep_row.to_numpy()).to(torch.float32).to(device)
    return dep_row_torch, indep_row_torch

city_vars_pt = [ i for i, ele in enumerate(indep_frd.columns) if 'city_' in ele and 'merchant_city_' not in ele and 'city_match' not in ele ]
mer_city_vars_pt = [ i for i, ele in enumerate(indep_frd.columns) if 'merchant_city_' in ele and 'city_match' not in ele ]
mcc_vars_pt = [ i for i, ele in enumerate(indep_frd.columns) if 'mcc_' in ele ]
add_indep_vars = [ i for i, ele in enumerate(indep_vars) if i not in city_vars_pt and i not in mer_city_vars_pt and i not in mcc_vars_pt ]

dep_row, indep_row = random_row(dep_frd, indep_frd, dep_not_frd, indep_not_frd)
dep_row_torch = torch.from_numpy(dep_row.to_numpy().squeeze(axis=1)).to(torch.long).to(device)
indep_row_torch = torch.from_numpy(indep_row.to_numpy()).to(torch.float32).to(device)
indep_row_torch[:, [0,1,2]]

def score_hidden(out1, out2):
    return out1.flatten().dot(out2.flatten()).reshape((1,1))
class FraudClassify(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super(FraudClassify, self).__init__()

        self.h_size = hidden_size
        self.o_size = output_size

        n_var_emb = 5

        self.l2l_add = nn.Linear(input_size[0], hidden_size)
        self.l2l_cty = nn.Linear(input_size[1], n_var_emb)
        self.l2l_mcty = nn.Linear(input_size[2], n_var_emb)
        self.l2l_mcc = nn.Linear(input_size[3], n_var_emb)

        self.l2h_add = nn.Linear(hidden_size, hidden_size)
        self.l2h_cty = nn.Linear(n_var_emb, hidden_size)
        self.l2h_mcty = nn.Linear(n_var_emb, hidden_size)
        self.l2h_mcc = nn.Linear(n_var_emb, hidden_size)

        mult = 4
        self.h2h = nn.Linear(hidden_size * mult + 6, hidden_size * mult)
        self.hi2n = nn.Linear(hidden_size * mult, hidden_size * mult)
        self.norm_one = nn.LayerNorm(hidden_size * mult)
        self.norm2norm = nn.Linear(hidden_size * mult, hidden_size * mult)
        self.norm_two = nn.LayerNorm(hidden_size * mult)
        self.norm2l = nn.Linear(hidden_size * mult * 3 + 6, hidden_size * 2)
        self.l2l = nn.Linear(hidden_size * 2, hidden_size)
        self.l2o = nn.Linear(hidden_size, output_size)
        self.lsoft = nn.LogSoftmax(dim=1)

    def forward(self, inpt_add, inpt_cty, inpt_mcty, inpt_mcc):

        out_add, out_cty, out_mcty, out_mcc = self.l2h_add(self.l2l_add(inpt_add)),\
                                              self.l2h_cty(self.l2l_cty(inpt_cty)),\
                                              self.l2h_mcty(self.l2l_mcty(inpt_mcty)),\
                                              self.l2h_mcc(self.l2l_mcc(inpt_mcc))
        a_c, a_mc, a_mcc = score_hidden(out_add, out_cty), score_hidden(out_add, out_mcty), score_hidden(out_add, out_mcc)
        m_mc, m_mcc = score_hidden(out_cty, out_mcty), score_hidden(out_cty, out_mcc)
        mc_mcc = score_hidden(out_mcty, out_mcc)

        feature_embeddings = torch.cat([out_add, out_cty, out_mcty, out_mcc], dim=1)
        scores = torch.cat([a_c, a_mc, a_mcc, m_mc, m_mcc, mc_mcc], dim=1)
        out = self.h2h(torch.cat([feature_embeddings, scores], dim=1))
        out = self.hi2n(out)
        hidden = self.norm_one(out)
        out = self.norm2norm(out)
        out = self.norm_two(out)
        out = self.norm2l(torch.cat([out, hidden, scores, feature_embeddings], dim=1))
        out = self.l2l(out)
        out = self.l2o(out)
        out = self.lsoft(out)

        return out

def train_a_epoch(model, criterion, optimizer, dep_frd, indep_frd, dep_not_frd, indep_not_frd):
    optimizer.zero_grad()
    flag = True
    while flag:
        try:
            dep_row_torch, indep_row_torch = random_training_pair(dep_frd, indep_frd, dep_not_frd, indep_not_frd)
            out = model.forward(indep_row_torch[:,add_indep_vars],\
                                indep_row_torch[:,city_vars_pt],\
                                indep_row_torch[:,mer_city_vars_pt],\
                                indep_row_torch[:,mcc_vars_pt])
            loss = criterion(out, dep_row_torch)
            loss.backward()
            optimizer.step()
            flag = False
        except:
            pass
    return out, loss.item()


In [ ]:
MODEL_PATH = './fraud_classify_score'

hidden_size = 30 # indep_frd.shape[1]
model = FraudClassify((len(add_indep_vars), len(city_vars_pt), len(mer_city_vars_pt), len(mcc_vars_pt)), hidden_size, 2).to(device)
# model.load_state_dict(torch.load(MODEL_PATH))
# model.train()
# model.eval()
model = model.to(device)

hidden = torch.zeros(1, hidden_size)
train_losses = []
print(f'num model parameters: {sum(p.numel() for p in model.parameters())}')

In [22]:
# optimizer = torch.optim.SGD(model.parameters(), lr=4e-2, momentum=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

def run(dep_frd, indep_frd, dep_not_frd, indep_not_frd, iter_num=1000):
    global model, hidden, optimizer, criterion, train_losses
    for iter in range(1, iter_num+1):

        model.train()
        epoch_out, epoch_loss = train_a_epoch(model, criterion, optimizer, dep_frd, indep_frd, dep_not_frd, indep_not_frd)
        train_losses.append(epoch_loss)
        print(f'{iter} epoch_loss: {epoch_loss}')
        model.eval()
        if iter % 100000 == 0:
            torch.save(model.state_dict(), MODEL_PATH)
            torch.save(hidden, f'./{MODEL_PATH}_hidden.pt')

    print("****************** training finished ******************")
    return model, train_losses

In [ ]:
model, train_losses = run(dep_frd, indep_frd, dep_not_frd, indep_not_frd, iter_num=3000000)